Preamble


In [ ]:
%reload_ext autoreload
%autoreload 2

from mplc import MPLC
from mplc.data import SICFieldDataset, get_basis_fields, RandomFieldDataset
import mplc.functional as mplcF
import torch

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

Define MPLC specs

In [ ]:
max_size = 0.01 # width of the phase masks ; m
beam_waist = 1e-3 # beam waist of the laser ; m
wavelength = 810e-9 # wavelength of laser ; m
spacing = 5e-6 #  size of the pixels used in the simulation ; m

shape = int(max_size//spacing) # number of pixels to use, decided by the size and spacing
print('Initializing', shape, 'pixels.')

mask_spacing = 5e-5 # size of the phase mask pixels (greater or equal to the size of the simulation pixels) ; m
num_masks = 5 # number of masks
propagation_distance = 0.09 # distance between each mask ; m


device = 'cpu' # device on which you want to train the network
dtype = torch.complex128 # to avoid type errors, generally stick with torch.complex128

Define the target gate, and create an associated dataset.

In [ ]:
dimension = 4 # dimension of the target gate
target = mplcF.hadamard(dimension) # target gate in matrix form
separation = 1.2 # separation of the logical basis fields: |k_j> = |separation*j>
basis_fields = get_basis_fields(dimension, shape, spacing, beam_waist, device, separation) # list of fields specifying the logical basis. transverse k-modes in our implementation
dataset = SICFieldDataset(target, basis_fields)  # the minimal, symmetric dataset which completely specifies the target gate
batch_size = 16 # since our dataset only has 16 elements in this example, we can use all the datapoints each epoch. For larger datasets, keep this smaller. 
dataloader = dataset.get_dataloader(batch_size) # this object passes the dataset to the model with the appropriate batching, shuffling, etc.

Create a model with the specs

In [ ]:
smooth_weight = 0.002 # how harshly should the training penalize a large second derivative
init_scale = 1e-1 # the magnitude scale of the initial phase mask parameters


torch.manual_seed(2) # some seeds are luckier than others, especially for a large init_scale. if you're having trouble with convergence, try a few different seeds
model = MPLC(shape, spacing, mask_spacing, num_masks, propagation_distance, beam_waist, wavelength, device, smooth_weight, init_scale)

params_list = [{'params': model.phase_masks.parameters(), 'lr': 0.17, 'weight_decay': 0.025}]
optimizer = torch.optim.AdamW(params_list)
loss_func = mplcF.field_overlap_loss # given by 1- avg(|< psi_target | psi_output >|)

In [ ]:
num_epochs = 200
model.train_mplc(num_epochs, dataloader, optimizer, device, loss_func)

Plot the training curve and phase masks

In [ ]:
a = model.visualize_model(show = False)

Plot the propagation of each logical basis state through the MPLC, in either the near field or the far field.

In [ ]:
fig = model.visualize_field(basis_fields, far_field=True)

Plot the phase gradient across each mask, comparing it to the maximum of $\frac{\pi\;rad}{50\mu m}$.

In [ ]:
model.vizualize_phase_gradient()

The best figure of merit is the average performance across any states, not just SIC-POVM or logical states. 

In [ ]:
test_dataset = RandomFieldDataset(target, basis_fields, 1000) # 1000 states!
test_dataloader = test_dataset.get_dataloader(60) # don't use all the states at once, it will be very slow

test_loss = model.test(test_dataloader, loss_func)
print(1-test_loss)

Save your masks.

In [ ]:
model.export_masks("your/folder/name/here")